Лабораторна робота №2
Частина 2

Встановила офіційну бібліотеку репозиторію UCI командою pip install ucimlrepo.
Через запропонований скрипт на сайті завантажую Individual Household Electric Power Consumption Dataset.

In [2]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd
import numpy as np
import timeit

# завантажуємо датасет
dataset = fetch_ucirepo(id=235) 

# об'єднуємо в один датафрейм
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

# на майбутнє, щоб було зручно виконувати одне завдання, переведу стовпчик Time у тип string
df['Time'] = df['Time'].astype(str)

print(f"Дані успішно завантажено. Кількість записів: {len(df)}")
display(df.head())

Дані успішно завантажено. Кількість записів: 2075259


/Users/margarita/зпад/venv/lib/python3.13/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (0: Global_active_power, 1: Global_reactive_power, 2: Voltage, 3: Global_intensity, 4: Sub_metering_1, 5: Sub_metering_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


Тепер створю функцію для очистки даних.

In [3]:
import pandas as pd
import numpy as np

def clean_data(df):
    # замінюємо '?' на NaN, тому що в цьому датасеті саме так позначаються пропущені дані
    df = df.replace('?', np.nan)
    
    # підраховує, скільки пропущених значень
    missing_count = df.isnull().sum()
    print("Пропущених даних по стовпчикам:\n", missing_count)
    
    # видаляємо рядки з пропусками
    df_cleaned = df.dropna().copy()
    
    # Конвертуємо тип всіх стовпчиків, крім Date та Time, на числове значення float
    cols_to_convert = df_cleaned.columns.drop(['Date', 'Time'])
    for col in cols_to_convert:
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')
    
    # видаляємо рядки, які могли невдало конвертуватись 
    df_cleaned = df_cleaned.dropna()
    
    print(f"Кількість записів після очищення: {len(df_cleaned)}")
    print(f"Видалено рядків: {len(df) - len(df_cleaned)}")
    
    return df_cleaned


In [4]:
# виклик функції
df = clean_data(df)

Пропущених даних по стовпчикам:
 Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64
Кількість записів після очищення: 2049280
Видалено рядків: 25979


25979 видалених рядків це 1.25% від загальної кількості. Це ідельний результат, тому що в самій документації до датасету вказано, що пропущених значень саме таке число.

Наступним завданням було сформувати наступні вибірки окремими функціями:

1) Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [5]:
import timeit # імпортую цей модуль для того, щоб здійснити профілювання часу

def high_power_records(data):
    return data[data['Global_active_power'] > 5]

# вимірюємо час виконання за допомогою модуля timeit
execution_time = timeit.timeit(lambda: high_power_records(df), number=1)
task1 = high_power_records(df)

print(f"Кількість знайдених записів: {len(task1)}")
print(f"Час виконання процедури: {execution_time:.5f} секунд")

Кількість знайдених записів: 17547
Час виконання процедури: 0.00688 секунд


In [6]:
# подивитись п'ять перших записів
display(task1.head())

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


2) Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.

Відповідно до документації:
sub_metering_2 - пралка та холодильник
sub_metering_3 - бойлер та кондиціонер

In [7]:
import timeit

def global_intensity_comparison(data):
    return data[(data['Global_intensity'] >= 19) & # фільтруємо силу струму від 19 до 20 А
                (data['Global_intensity'] <= 20) & 
                (data['Sub_metering_2'] > data['Sub_metering_3'])] # порівнюємо споживання (де пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер)

# вимірюємо час виконання
execution_time = timeit.timeit(lambda: global_intensity_comparison(df), number=1)

task2 = global_intensity_comparison(df)

print(f"Кількість знайдених записів: {len(task2)}")
print(f"Час виконання процедури: {execution_time:.5f} секунд")


Кількість знайдених записів: 2509
Час виконання процедури: 0.00641 секунд


In [8]:
# переглянути перші п'ять записів
display(task2.head())

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


3) Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [9]:
import timeit

def random_sample_means(data, n_samples=500000):
    # вибираємо випадкові записи без повторів
    sample = data.sample(n=n_samples, replace=False, random_state=42)
    
    # обчислюємо середнє для трьох стовпців споживання
    means = sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means

# вимірюємо час виконання
execution_time = timeit.timeit(lambda: random_sample_means(df), number=1)

result_means = random_sample_means(df)

print(f"Час виконання: {execution_time:.5f} секунд")
print("\nСередні значення для 3-х груп споживання (Вт-год):")
print(result_means)

Час виконання: 0.11012 секунд

Середні значення для 3-х груп споживання (Вт-год):
Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
dtype: float64


4) Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [10]:
import pandas as pd
import timeit

def complex_selection(data):
    # фільтруємо за часом (> 18:00) та потужністю (> 6 кВт)
    # для цього я Time перетворювала на string, так набагато зручніше
    condition_1 = (data['Time'] > '18:00:00') & (data['Global_active_power'] > 6)
    subset = data[condition_1].copy()
    
    # визначаємо, де Група 2 (Sub_metering_2) є найбільшою серед усіх трьох
    condition_2 = (subset['Sub_metering_2'] > subset['Sub_metering_1']) & \
                  (subset['Sub_metering_2'] > subset['Sub_metering_3'])
    final_subset = subset[condition_2]
    
    # ділимо на половину
    n = len(final_subset)
    mid = n // 2
    
    first_half = final_subset.iloc[:mid:3]   # кожен третій результат з першої половини
    second_half = final_subset.iloc[mid::4] # кожен четвертий з другої
    
    return pd.concat([first_half, second_half])

# вимірюємо час
execution_time = timeit.timeit(lambda: complex_selection(df), number=1)
task4 = complex_selection(df)

print(f"Знайдено записів: {len(task4)}")
print(f"Час виконання процедури: {execution_time:.5f} секунд")

Знайдено записів: 310
Час виконання процедури: 0.07086 секунд


In [11]:
# подивитись перші п'ять записів
display(task4.head())

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


Пронормувати та стандартизувати вибраний датасет.
Це важливий етап, оскільки усі стовпці мають різний маштаб. Нормалізація зводить усі значення до діапазону [0,1]. Стандартизація 
зсуває дані так, щоб середнє значення дорівнювало 0, а стандартне відхилення 1.

In [12]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# вибираємо лише числові колонки для масштабування
columns = df.columns.drop(['Date', 'Time'])

# нормалізуємо
scaler_minmax = MinMaxScaler()
df_normalized = df.copy()
df_normalized[columns] = scaler_minmax.fit_transform(df[columns])

# стандартизуємо
scaler_std = StandardScaler()
df_standardized = df.copy()
df_standardized[columns] = scaler_std.fit_transform(df[columns])

print("Все завершено успішно.")

Все завершено успішно.


In [13]:
print("Статистика після нормалізації (Global_active_power):")
print(f"Min: {df_normalized['Global_active_power'].min()}, Max: {df_normalized['Global_active_power'].max()}")

print("\nСтатистика після стандартизації (Global_active_power):")
print(f"Mean: {df_standardized['Global_active_power'].mean():.2f}, Std: {df_standardized['Global_active_power'].std():.2f}")

Статистика після нормалізації (Global_active_power):
Min: 0.0, Max: 1.0

Статистика після стандартизації (Global_active_power):
Mean: -0.00, Std: 1.00


In [14]:
!pip install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.
Коефіцієнт Пірсона коливається від -1 до 1. Цей коефіцієнт показує, наскільки близько точки даних на діаграмі розсіювання розташовані до прямої лінії.
Коефіцієнт Спірмена вимірює монотонний зв'язок. Він працює з рангами (порядком) значень. Він "стійкіший", якщо зв'язок між показниками є криволінійним або в даних є аномалії.

In [19]:
attr1 = 'Global_active_power' # загальна активна потужність
attr2 = 'Global_intensity' # сила струму

# Розрахунок коефіцієнта Пірсона
pearson_corr = df[attr1].corr(df[attr2], method='pearson')

# Розрахунок коефіцієнта Спірмена
spearman_corr = df[attr1].corr(df[attr2], method='spearman')

print(f"Аналіз взаємозв'язку між '{attr1}' та '{attr2}':")
print(f"Коефіцієнт Пірсона: {pearson_corr:.4f}")
print(f"Коефіцієнт Спірмена: {spearman_corr:.4f}")

Аналіз взаємозв'язку між 'Global_active_power' та 'Global_intensity':
Коефіцієнт Пірсона: 0.9989
Коефіцієнт Спірмена: 0.9954


Коефіцієнти 0.9989 та 0.9954 свідчать про майже ідеальну пряму залежність. Це логічно, бо потужність дорівнює напрузі, помноженій на силу струму. Оскільки напруга в розетці коливається слабо, то потужність і сила струму змінюються практично синхронно.

Провести One Hot Encoding категоріального атрибута.

In [22]:
import pandas as pd

# перетворюю стовпець Date у формат дати Python
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# створюю категоріальний атрибут (наприклад, назва дня тижня)
df['Day_of_week'] = df['Date'].dt.day_name()

# One Hot Encoding за допомогою pandas.get_dummies
# це створює стовпці: Day_of_week_Monday, Day_of_week_Tuesday і т.д.
df_with_ohe = pd.get_dummies(df, columns=['Day_of_week'], prefix='Day')

print("One Hot Encoding завершено")
print(f"Нова кількість стовпців: {len(df_with_ohe.columns)}")
display(df_with_ohe.head())

One Hot Encoding завершено
Нова кількість стовпців: 16


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Day_Friday,Day_Monday,Day_Saturday,Day_Sunday,Day_Thursday,Day_Tuesday,Day_Wednesday
0,2006-12-16,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,False,False,True,False,False,False,False
1,2006-12-16,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,False,False,True,False,False,False,False
2,2006-12-16,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,False,False,True,False,False,False,False
3,2006-12-16,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,False,False,True,False,False,False,False
4,2006-12-16,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,False,False,True,False,False,False,False
